# SLA, Capacity, and Commercials

Riverside gives you 620 sessions per business day, a high case of 900, and a peak of 90 requests per hour. A spreadsheet can divide 620 by ten covered hours, but that produces sessions per hour, not requests per hour, and it hides bursts.

Your job is not to produce the neatest number. Your job is to show Riverside what can be planned now, what needs measurement, and what cannot yet be promised.

Throughout the notebook:

- `[Measured: synthetic fixture]` means a committed synthetic trace was observed.
- `[Modeled]` means code calculated from explicit assumptions.
- `[Policy constraint]` means cost or availability cannot trade it away.
- `[External validation required]` means a named owner still owes live evidence or approval.

A modeled result stays modeled even after Python prints it.


## 0 - Riverside wants a tier, but the inputs do not agree

Riverside asks for a 99.5% service target, weekday support, deadline help when needed, a 14,000 USD monthly target, and an 18,000 USD planning ceiling. At the same time, the demand inputs mix sessions and requests, and live prices, quota, failover, and staffing are unknown.

```mermaid
flowchart LR
    A["Demand, latency, budget,<br/>support inputs"] --> B["Reject average-only plan"]
    B --> C["Build low / expected / high stories"]
    C --> D["Check independent bottlenecks"]
    D --> E["Price the operated service"]
    E --> F["Pilot, narrow, or seek approval"]
```

**Predict before running:** Which gap blocks a final quote first: conflicting demand units, placeholder prices, unknown quota, or unapproved support commitments? More than one answer may be correct; name the owner for each.

Forbidden access and duplicate workflow commits remain zero-tolerance policy constraints. They do not become acceptable because the availability percentage looks good.


In [ ]:
# -- Load frozen and measured synthetic evidence ---------------------------
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        path
        for path in candidates
        if (path / "AUTHORING_GUIDE.md").is_file()
        and (path / "learning" / "role-based-tracks" / "fde" / "shared").is_dir()
    ),
    None,
)
assert REPO_ROOT is not None, "Run from inside the ai-portfolio repository."
CHAPTER_DIR = REPO_ROOT / "learning" / "role-based-tracks" / "fde" / "05-sla-capacity-and-commercials"
CASE_PATH = REPO_ROOT / "learning" / "role-based-tracks" / "fde" / "shared" / "fixtures" / "riverside-engagement-v1.json"
FACTS_PATH = REPO_ROOT / "learning" / "role-based-tracks" / "fde" / "shared" / "fixtures" / "expected-facts-v1.json"
TRACES_PATH = REPO_ROOT / "learning" / "ai-engineer" / "shared" / "latency-cost" / "request-traces.jsonl"
COST_INPUT_PATH = CHAPTER_DIR / "templates" / "cost-input-template.csv"

case = json.loads(CASE_PATH.read_text(encoding="utf-8"))
expected_facts = json.loads(FACTS_PATH.read_text(encoding="utf-8"))
traces = [json.loads(line) for line in TRACES_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assumption_rows = case["demand_cost_and_sla_assumptions"]["assumptions"]
assumptions = {row["name"]: row for row in assumption_rows}
service_targets = case["demand_cost_and_sla_assumptions"]["service_targets"]
fact_by_id = {fact["fact_id"]: fact for fact in expected_facts["facts"]}

successful = [trace for trace in traces if trace["outcome"] == "success"]
trace_cost = sum(trace["total_cost_microusd"] for trace in traces) / 1_000_000
trace_attempts = sum(stage["stage_name"] == "generation" for trace in traces for stage in trace["stages"])
trace_generation_requests = sum(any(stage["stage_name"] == "generation" for stage in trace["stages"]) for trace in traces)

checks = {
    "fixture_version_matches": expected_facts["fixture_version"] == case["fixture_version"],
    "expected_sessions_preserved": assumptions["average_daily_sessions"]["value"] == fact_by_id["FACT-RIV-019"]["expected_value"],
    "budget_preserved": assumptions["monthly_service_budget_target"]["value"] == fact_by_id["FACT-RIV-021"]["expected_value"],
    "all_demand_inputs_modeled": all(row["evidence_class"] == "modeled_assumption" for row in assumption_rows),
    "trace_ids_unique": len({trace["request_id"] for trace in traces}) == len(traces),
}
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(checks.values()), "Stop: source evidence failed integrity checks."
print(f"[Modeled input] frozen fixture: {case['fixture_version']}")
print(f"[Measured: synthetic fixture] requests: {len(traces)}, successful: {len(successful)}, observed cost: ${trace_cost:.6f}")
print(f"[Measured: synthetic fixture] generation-attempt amplification: {trace_attempts / trace_generation_requests:.2f}x")
print("LIMIT: five synthetic traces teach attribution; they do not estimate production percentiles or price.")

## 1 - Break the average-only plan

A quick plan says:

> 620 sessions / 10 covered hours = 62 sessions per hour, so size for 62 requests per hour.

The last step silently changes the unit from sessions to requests. Riverside also states a 90-request-per-hour peak. If the expected session contains several requests, the implied average can even exceed that stated peak. The inputs need reconciliation before sizing.

```mermaid
flowchart LR
    A["620 sessions/day"] --> B["62 sessions/hour"]
    B --> C["Hidden guess:<br/>1 request/session"]
    C --> D["Wrongly called 62 requests/hour"]
    E["Stated peak:<br/>90 requests/hour"] --> F["Units conflict"]
    D --> F
    F --> G["Use larger safe planning input<br/>and assign validation owner"]
```

**Predict before running:** Which checks will fail: the stated peak versus the one-request average, the stated peak versus the chapter's expected requests per session, or both?

**Before:** one smooth daily average becomes a capacity promise.

**After:** keep sessions and requests separate, expose the conversion assumption, and use the larger conflicting rate until Riverside supplies measured traffic.

**Plain takeaway:** an average is a reasonableness check, not a burst plan.


In [ ]:
# ── Expose and Plot the Average-Only Failure ──────────────────────────────────
covered_hours_per_day = 10
expected_sessions = assumptions["average_daily_sessions"]["value"]
stated_peak_rph = assumptions["peak_arrival_rate"]["value"]
chapter_requests_per_session = 3.4  # [Modeled] Exercise input; not a frozen customer fact.

naive_rph = expected_sessions / covered_hours_per_day
implied_rph = expected_sessions * chapter_requests_per_session / covered_hours_per_day
planning_rph = max(stated_peak_rph, implied_rph)
print(f"[Modeled] naive one-request/session average: {naive_rph:.1f} requests/hour")
print(f"[Modeled] frozen stated peak: {stated_peak_rph:.1f} requests/hour")
print(f"[Modeled] implied average at {chapter_requests_per_session:.1f} requests/session: {implied_rph:.1f} requests/hour")
print(f"[Modeled] conservative planning rate before reconciliation: {planning_rph:.1f} requests/hour")
if implied_rph > stated_peak_rph:
    prediction_result = "prediction 3 confirmed - the inputs use unreconciled demand definitions"
    next_action = "size from the larger rate temporarily and assign Operations to reconcile telemetry"
else:
    prediction_result = "prediction 1 confirmed - the frozen peak exceeds the modeled implied average"
    next_action = "retain the frozen peak provisionally and validate requests per session plus burst shape"
print(f"RESULT: {prediction_result}.")
print(f"ACTION: {next_action}.")

labels = ["Naive average", "Frozen peak", "Implied average", "Planning rate"]
values = [naive_rph, stated_peak_rph, implied_rph, planning_rph]
colors = ["#1d4ed8", "#b45309", "#b91c1c", "#15803d"]
fig, ax = plt.subplots(figsize=(9, 4.5), facecolor="#1a1a2e")
ax.set_facecolor("#1a1a2e")
bars = ax.bar(labels, values, color=colors)
ax.bar_label(bars, fmt="%.1f", color="white")
ax.set_ylabel("Requests per hour", color="white")
ax.set_title("[Modeled] One average cannot reconcile session and request demand", color="white")
ax.tick_params(colors="white", axis="x", rotation=15)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

average_plan_checks = {
    "positive_covered_hours": covered_hours_per_day > 0,
    "positive_requests_per_session": chapter_requests_per_session > 0,
    "planning_rate_not_below_any_input": planning_rph >= max(stated_peak_rph, implied_rph),
    "reconciliation_flag_matches_inputs": (implied_rph > stated_peak_rph) == ("unreconciled" in prediction_result),
}
for name, passed in average_plan_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert average_plan_checks["planning_rate_not_below_any_input"]
print("LIMIT: this conservative maximum is a temporary planning rule, not a measured arrival distribution.")

## 2 - Build three understandable demand stories

Low, expected, and high are not probabilities. They are three complete Riverside stories:

- **Low:** fewer sessions, shorter requests, fewer retries, and bounded cache reuse.
- **Expected:** the current planning assumptions.
- **High:** more sessions, heavier request mix, more retries, and less realized cache benefit.

Each story must use consistent units and keep its assumptions visible.

```mermaid
flowchart TD
    A["Sessions and requests/session"] --> D["Monthly requests"]
    B["Request mix and token sizes"] --> E["Monthly tokens"]
    C["Retries and safe cache realization"] --> D
    C --> E
    D --> F["Capacity and cost"]
    E --> F
```

**Predict before running:** Which high-case change will amplify both capacity and cost: sessions, requests per session, retries, cache realization, or token size?

Do not label these stories p10, p50, or p90; no probability model was fitted. Cache eligibility is not an observed hit rate, and cache keys must preserve authorization scope.

**Plain takeaway:** a scenario is an explicit story, not a confidence interval.


In [ ]:
# -- Build, plot, and check demand/token scenarios -------------------------
request_mix = assumptions["request_mix"]["value"]
business_days_per_month = 22
scenario_inputs = {
    "low": {"sessions_per_day": 450, "stated_peak_rph": 60, "requests_per_session": 2.5, "input_tokens": 900, "output_tokens": 90, "cache_eligible": 0.08, "cache_realization": 0.45, "retry_rate": 0.01, "support_hours": 48, "budget": 11000, "headroom": 0.25, "policy_s": 3.5, "continuation_s": 8.0, "workflow_s": 5.0},
    "expected": {"sessions_per_day": 620, "stated_peak_rph": 90, "requests_per_session": 3.4, "input_tokens": 1800, "output_tokens": 220, "cache_eligible": 0.18, "cache_realization": 0.65, "retry_rate": 0.025, "support_hours": 80, "budget": 14000, "headroom": 0.35, "policy_s": 5.0, "continuation_s": 14.0, "workflow_s": 8.0},
    "high": {"sessions_per_day": 900, "stated_peak_rph": 150, "requests_per_session": 5.0, "input_tokens": 4200, "output_tokens": 600, "cache_eligible": 0.28, "cache_realization": 0.35, "retry_rate": 0.07, "support_hours": 140, "budget": 18000, "headroom": 0.50, "policy_s": 8.0, "continuation_s": 18.0, "workflow_s": 12.0},
}

scenario_rows = []
for name, values in scenario_inputs.items():
    monthly_requests = values["sessions_per_day"] * values["requests_per_session"] * business_days_per_month
    implied_average_rph = values["sessions_per_day"] * values["requests_per_session"] / covered_hours_per_day
    conservative_rph = max(values["stated_peak_rph"], implied_average_rph)
    realized_hit_rate = values["cache_eligible"] * values["cache_realization"]
    billed_attempts = monthly_requests * (1 - realized_hit_rate) * (1 + values["retry_rate"])
    scenario_rows.append({**values, "scenario": name, "monthly_requests": monthly_requests, "implied_average_rph": implied_average_rph, "planning_rph": conservative_rph, "realized_hit_rate": realized_hit_rate, "billed_input_tokens": billed_attempts * values["input_tokens"], "billed_output_tokens": billed_attempts * values["output_tokens"]})

scenarios = pd.DataFrame(scenario_rows).set_index("scenario")
display(scenarios[["sessions_per_day", "monthly_requests", "stated_peak_rph", "implied_average_rph", "planning_rph", "realized_hit_rate", "retry_rate", "billed_input_tokens", "billed_output_tokens"]].round(2))
print("[Modeled] scenario outputs retain modeled status; no probability is assigned to low/expected/high.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor="#1a1a2e")
scenario_colors = ["#1e3a8a", "#1d4ed8", "#b45309"]
scenarios["monthly_requests"].plot.bar(ax=axes[0], color=scenario_colors)
axes[0].set_title("[Modeled] Monthly request range")
axes[0].set_ylabel("Requests")
(scenarios[["billed_input_tokens", "billed_output_tokens"]] / 1_000_000).plot.bar(ax=axes[1], color=["#1d4ed8", "#15803d"])
axes[1].set_title("[Modeled] Billed token range")
axes[1].set_ylabel("Million tokens")
for ax in axes:
    ax.set_facecolor("#1a1a2e")
    ax.tick_params(axis="x", rotation=0)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

scenario_checks = {
    "request_mix_sums_to_one": math.isclose(sum(request_mix.values()), 1.0),
    "sessions_are_ordered": scenarios.loc["low", "sessions_per_day"] < scenarios.loc["expected", "sessions_per_day"] < scenarios.loc["high", "sessions_per_day"],
    "tokens_are_positive": (scenarios[["input_tokens", "output_tokens"]] > 0).all().all(),
    "cache_rates_are_bounded": scenarios["realized_hit_rate"].between(0, 1).all(),
    "retry_rates_are_bounded": scenarios["retry_rate"].between(0, 1).all(),
    "planning_rate_covers_inputs": (scenarios["planning_rph"] >= scenarios[["stated_peak_rph", "implied_average_rph"]].max(axis=1)).all(),
}
for name, passed in scenario_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(scenario_checks.values()), "Stop: scenario model is internally inconsistent."

## 3 - Use simple concurrency intuition, then check every limit

Imagine requests as Riverside editors occupying service slots. If two requests arrive each second and each occupies a slot for three seconds, the steady-state floor is about six in-flight requests. That is the intuition behind arrival rate times time in system.

The notebook uses a request-mix-weighted planning time, includes retry and cache effects, and then adds headroom once. This floor is not a p95 guarantee. Bursts, long requests, and correlated retries still need a load test.

```mermaid
flowchart LR
    A["How fast work arrives"] --> D["In-flight floor"]
    B["How long work holds a slot"] --> D
    C["Retries and cache effects"] --> D
    D --> E["Add headroom once"]
    E --> F["Check RPM, TPM,<br/>concurrency, queue, spend"]
```

**Predict before running:** In the high case, will request quota or token quota be tighter? Write the expected utilization for each in words, then run the cell.

| Mistake | Better Riverside decision |
|---|---|
| Check RPM only | Check RPM and TPM; long prompts can exhaust tokens first |
| Add headroom twice | Record where headroom enters and apply it once |
| Call the floor a tail guarantee | Treat it as a planning floor and schedule burst/load evidence |

**Plain takeaway:** capacity is whichever independent limit is reached first.


In [ ]:
# -- Model, plot, and check independent capacity dimensions ---------------
quota = {"rpm": 120, "tpm": 300_000, "concurrency": 20}  # [Modeled] Placeholder, not provider evidence.
capacity_rows = []
for name, row in scenarios.iterrows():
    uncached_service_s = request_mix["policy_lookup"] * row["policy_s"] + request_mix["editorial_continuation"] * row["continuation_s"] + request_mix["workflow_assistance"] * row["workflow_s"]
    effective_service_s = row["realized_hit_rate"] * 0.1 + (1 - row["realized_hit_rate"]) * uncached_service_s * (1 + row["retry_rate"])
    arrival_rps = row["planning_rph"] / 3600
    concurrency_floor = arrival_rps * effective_service_s
    planned_concurrency = max(1, math.ceil(concurrency_floor * (1 + row["headroom"])))
    peak_rpm = row["planning_rph"] / 60
    peak_tpm = peak_rpm * (1 - row["realized_hit_rate"]) * (1 + row["retry_rate"]) * (row["input_tokens"] + row["output_tokens"])
    capacity_rows.append({"scenario": name, "effective_service_s": effective_service_s, "concurrency_floor": concurrency_floor, "planned_concurrency": planned_concurrency, "peak_rpm": peak_rpm, "peak_tpm": peak_tpm, "rpm_utilization": peak_rpm / quota["rpm"], "tpm_utilization": peak_tpm / quota["tpm"], "concurrency_utilization": planned_concurrency / quota["concurrency"]})

capacity = pd.DataFrame(capacity_rows).set_index("scenario")
display(capacity.round(3))
high_driver = capacity.loc["high", ["rpm_utilization", "tpm_utilization", "concurrency_utilization"]].idxmax()
print(f"RESULT: high-case illustrative limiting dimension is {high_driver}.")
print("[External validation required] Replace all three quota placeholders with approved route/region evidence.")

utilization = capacity[["rpm_utilization", "tpm_utilization", "concurrency_utilization"]] * 100
ax = utilization.plot.bar(figsize=(10, 4.5), color=["#1d4ed8", "#b45309", "#15803d"])
ax.set_facecolor("#1a1a2e")
ax.figure.set_facecolor("#1a1a2e")
ax.axhline(100, color="#b91c1c", linestyle="--", label="Illustrative quota")
ax.set_title("[Modeled] Independent quota utilization")
ax.set_ylabel("Utilization (%)")
ax.tick_params(axis="x", rotation=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

capacity_checks = {
    "planned_concurrency_covers_floor": (capacity["planned_concurrency"] >= capacity["concurrency_floor"]).all(),
    "headroom_is_explicit": scenarios["headroom"].between(0, 1).all(),
    "token_rate_positive": (capacity["peak_tpm"] > 0).all(),
    "quota_dimensions_present": set(quota) == {"rpm", "tpm", "concurrency"},
}
for name, passed in capacity_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(capacity_checks.values())
print("LIMIT: queue depth, p99, cold starts, correlated retries, and burst duration need a load test or queueing simulation.")

## 4 - Price the service Riverside actually operates

A token estimate is not a service estimate. Riverside also pays for application infrastructure, retrieval and storage, monitoring and evaluation, software, support engineering, failed attempts, and idle capacity held for peaks.

```mermaid
flowchart TD
    A["Model tokens"] --> G["Monthly operated-service estimate"]
    B["Application infrastructure"] --> G
    C["Retrieval and storage"] --> G
    D["Monitoring and evaluation"] --> G
    E["Other software"] --> G
    F["Support people"] --> G
    G --> H["Separate uncertainty reserve"]
    H --> I["Compare with 14k target<br/>and 18k ceiling"]
```

**Predict before running:** If token rates fall by half but support and fixed infrastructure do not change, will total service cost also fall by half?

**Before:** only successful model calls are priced.

**After:** every component appears once with unit, currency, source date, exclusions, and validation owner. Failed and retried calls remain billable where applicable. The uncertainty reserve stays visible instead of being hidden inside infrastructure.

Security controls are never removed to make the spreadsheet fit.


In [ ]:
# -- Attribute, plot, and reconcile monthly service cost ------------------
cost_inputs = pd.read_csv(COST_INPUT_PATH)
assert (cost_inputs["validation_status"] == "external_validation_required").all()

def rate_for(scenario, component):
    match = cost_inputs[(cost_inputs["scenario"] == scenario) & (cost_inputs["cost_component"] == component)]
    assert len(match) == 1, f"Expected one rate for {scenario}/{component}"
    return float(match.iloc[0]["rate"])

cost_rows = []
for name, row in scenarios.iterrows():
    model_input = row["billed_input_tokens"] / 1_000_000 * rate_for(name, "model_input")
    model_output = row["billed_output_tokens"] / 1_000_000 * rate_for(name, "model_output")
    components = {
        "model": model_input + model_output,
        "application_infrastructure": rate_for(name, "application_infrastructure"),
        "retrieval_and_storage": rate_for(name, "retrieval_and_storage"),
        "observability_and_evaluation": rate_for(name, "observability_and_evaluation"),
        "other_software": rate_for(name, "other_software"),
        "support_engineering": row["support_hours"] * rate_for(name, "support_engineering"),
    }
    subtotal = sum(components.values())
    components["risk_reserve"] = subtotal * 0.10
    components["total"] = subtotal + components["risk_reserve"]
    components["budget"] = row["budget"]
    components["cost_per_request"] = components["total"] / row["monthly_requests"]
    cost_rows.append({"scenario": name, **components})

costs = pd.DataFrame(cost_rows).set_index("scenario")
display(costs.round(2))
for name, row in costs.iterrows():
    status = "WITHIN" if row["total"] <= row["budget"] else "EXCEEDS"
    print(f"[Modeled] {name}: ${row['total']:,.0f}/month {status} scenario budget ${row['budget']:,.0f}; not a quote.")

component_columns = ["model", "application_infrastructure", "retrieval_and_storage", "observability_and_evaluation", "other_software", "support_engineering", "risk_reserve"]
ax = costs[component_columns].plot.bar(stacked=True, figsize=(11, 5), colormap="tab20")
ax.set_facecolor("#1a1a2e")
ax.figure.set_facecolor("#1a1a2e")
ax.scatter(range(len(costs)), costs["budget"], color="#b91c1c", marker="_", s=700, linewidths=3, label="Scenario budget")
ax.axhline(18_000, color="white", linestyle="--", label="Hard planning ceiling")
ax.set_title("[Modeled] Monthly service cost range - illustrative rates")
ax.set_ylabel("USD per month")
ax.tick_params(axis="x", rotation=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

recomputed_total = costs[component_columns].sum(axis=1)
cost_checks = {
    "totals_reconcile": np.allclose(recomputed_total, costs["total"]),
    "all_rates_have_sources": cost_inputs["source_or_basis"].notna().all(),
    "all_rates_have_dates": cost_inputs["source_date"].notna().all(),
    "all_rates_have_validation_owner": cost_inputs["validation_owner"].notna().all(),
}
for name, passed in cost_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(cost_checks.values())
print("LIMIT: taxes, discounts, FX, egress, service credits, incident surge, and customer-side cost remain excluded.")

## 5 - Find the assumption that changes the decision

Riverside does not need a long list of possible optimizations. It needs to know which uncertain input can move the monthly result enough to change the pilot or budget decision.

The next cell starts from the expected case, changes one driver by 25% down and up, and recomputes cost. This is a before/after comparison, not a probability forecast.

```mermaid
flowchart LR
    A["Expected case"] --> B["One driver -25%"]
    A --> C["One driver +25%"]
    B --> D["Recompute monthly total"]
    C --> D
    D --> E["Rank cost movement"]
    E --> F["Validate the top uncertain driver"]
```

**Predict before running:** Which will move the total most: demand, token size, model rate, support cost, or cache realization?

Do not call this a confidence interval. It changes one input at a time and does not model interactions. Any cheaper option still has to meet quality, identity, isolation, and support floors. Increasing cache without authorization-scoped keys is not an optimization; it is a security defect.


In [ ]:
# -- Compute, plot, and check one-way sensitivity -------------------------
expected = scenario_inputs["expected"].copy()
expected_rates = {component: rate_for("expected", component) for component in cost_inputs["cost_component"].unique()}

def expected_total(values, rates):
    requests = values["sessions_per_day"] * values["requests_per_session"] * business_days_per_month
    hit_rate = values["cache_eligible"] * values["cache_realization"]
    attempts = requests * (1 - hit_rate) * (1 + values["retry_rate"])
    model = attempts * (values["input_tokens"] * rates["model_input"] + values["output_tokens"] * rates["model_output"]) / 1_000_000
    subtotal = model + rates["application_infrastructure"] + rates["retrieval_and_storage"] + rates["observability_and_evaluation"] + rates["other_software"] + values["support_hours"] * rates["support_engineering"]
    return subtotal * 1.10

baseline_total = expected_total(expected, expected_rates)
drivers = ["support_hours", "requests_per_session", "input_tokens", "output_tokens", "retry_rate", "cache_realization", "model_input_rate", "support_hourly_rate", "application_infrastructure"]
sensitivity_rows = []
for driver in drivers:
    totals = {}
    for label, factor in [("minus_25", 0.75), ("plus_25", 1.25)]:
        values = expected.copy()
        rates = expected_rates.copy()
        if driver == "model_input_rate":
            rates["model_input"] *= factor
        elif driver == "support_hourly_rate":
            rates["support_engineering"] *= factor
        elif driver == "application_infrastructure":
            rates["application_infrastructure"] *= factor
        else:
            values[driver] *= factor
        totals[label] = expected_total(values, rates)
    sensitivity_rows.append({"driver": driver, "minus_25_delta": totals["minus_25"] - baseline_total, "plus_25_delta": totals["plus_25"] - baseline_total, "swing": totals["plus_25"] - totals["minus_25"]})

sensitivity = pd.DataFrame(sensitivity_rows).set_index("driver").sort_values("swing")
display(sensitivity.round(2))
print(f"[Modeled] baseline: ${baseline_total:,.0f}/month; largest local swing: {sensitivity['swing'].idxmax()}")

fig, ax = plt.subplots(figsize=(10, 5.5), facecolor="#1a1a2e")
ax.set_facecolor("#1a1a2e")
y = np.arange(len(sensitivity))
ax.barh(y, sensitivity["minus_25_delta"], color="#1d4ed8", label="Driver -25%")
ax.barh(y, sensitivity["plus_25_delta"], color="#b45309", label="Driver +25%")
ax.axvline(0, color="white", linewidth=1)
ax.set_yticks(y, sensitivity.index)
ax.set_xlabel("Change in modeled monthly cost (USD)")
ax.set_title("[Modeled] Local sensitivity around expected case")
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

sensitivity_checks = {
    "baseline_matches_cost_table": math.isclose(baseline_total, costs.loc["expected", "total"], rel_tol=1e-9),
    "drivers_are_unique": sensitivity.index.is_unique,
    "both_directions_present": sensitivity[["minus_25_delta", "plus_25_delta"]].notna().all().all(),
}
for name, passed in sensitivity_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(sensitivity_checks.values())
print("ACTION: validate support hours/rate and fixed infrastructure before negotiating token discounts.")

## 6 - Turn a percentage into an operating commitment

Riverside's 99.5% target is scoped to covered service hours, but the sponsor also asks for help whenever a deadline is at risk. Weekday staffing cannot honestly support an unbounded 24x7 promise. The conflict must be decided by the service and commercial owners.

```mermaid
flowchart LR
    A["Named user journey<br/>and covered hours"] --> B["Measured indicators<br/>and target"]
    B --> C["Architecture and quota"]
    C --> D["Monitoring and stop rules"]
    D --> E["Rollout and rollback"]
    E --> F["Funded support and escalation"]
    F --> G["Commercial tier"]
```

**Predict before running:** If support is funded only on weekdays, which tier should be rejected or narrowed?

A service tier must name the user journey, covered hours, denominator, exclusions, architecture, quota, monitoring, rollout, rollback, staffing, escalation, and price. Forbidden access and duplicate commits stay outside the ordinary availability error budget; one such event is not balanced by many successful requests.

**Plain takeaway:** an SLA percentage without an operating system and funded people is only a target.


In [ ]:
# -- Derive, map, and check the covered-hours SLA tiers -------------------
availability_target = next(target["target"] for target in service_targets if target["name"] == "monthly_availability")
covered_hours_per_month = 10 * 5 * 52 / 12
allowed_unavailable_minutes = covered_hours_per_month * 60 * (1 - availability_target)

sla_tiers = pd.DataFrame([
    {"tier": "Pilot", "architecture": "One approved active route; disable switch; manual rollback", "quota": "Named cohort caps and hard spend/token limits", "monitoring": "Dashboard plus daily review; measure p50/p95/p99", "rollout": "Shadow then 12-editor canary", "support": "Best effort inside frozen covered hours", "offer_status": "Offerable as non-contractual pilot after security gates"},
    {"tier": "Business Hours", "architecture": "Redundant stateless app path; approved restore and fallback design", "quota": "Tenant RPM/TPM/concurrency/spend limits with validated headroom", "monitoring": "Burn, queue, throttle, retry, latency, and cost alerts", "rollout": "Canary and ramp gates with rollback owner", "support": "Severity response during 08:00-18:00 UK weekdays", "offer_status": "Provisional; load, quota, fallback, and terms validation required"},
    {"tier": "Critical", "architecture": "Fault-domain isolation and tested failover for approved state paths", "quota": "Reserved capacity and priority/load-shed policy", "monitoring": "24x7 paging, burn-rate response, tested incident evidence", "rollout": "Change windows, freeze rules, re-enablement authority", "support": "Staffed on-call and escalation explicitly priced", "offer_status": "Not offerable from current evidence or support agreement"},
]).set_index("tier")

print(f"[Modeled target] 99.5% across {covered_hours_per_month:.1f} covered hours allows about {allowed_unavailable_minutes:.1f} unavailable minutes/month.")
print("LIMIT: this arithmetic does not define exclusions, denominator, credits, or legal enforceability.")
display(sla_tiers)

tier_dimensions = ["architecture", "quota", "monitoring", "rollout", "support", "offer_status"]
sla_checks = {
    "all_tier_dimensions_present": sla_tiers[tier_dimensions].notna().all().all(),
    "availability_target_preserved": availability_target == fact_by_id["FACT-RIV-024"]["expected_value"],
    "covered_hours_preserved": case["support_and_handoff"]["covered_hours"] == fact_by_id["FACT-RIV-031"]["expected_value"],
    "critical_tier_not_offerable": "Not offerable" in sla_tiers.loc["Critical", "offer_status"],
    "security_not_spent_as_error_budget": next(target["target"] for target in service_targets if target["name"] == "authorization_leakage") == 0,
}
for name, passed in sla_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(sla_checks.values())

## 7 - Break one assumption at a time

Each exercise changes one decision input and prints a check. A passing assertion means the calculation is internally consistent. It does not approve a price, tier, or contract.

```mermaid
flowchart LR
    A["Write your prediction"] --> B["Change one variable"]
    B --> C["Recompute"]
    C --> D["Check budget or capacity gate"]
    D --> E["Name quality/security floor"]
    E --> F["Record decision or evidence owner"]
```

For each next exercise:

1. Run the default once.
2. Change only the marked variable.
3. Predict the budget or capacity result before rerunning.
4. Explain what measured evidence and approval would justify using the cheaper setting.
5. Reject any setting that weakens an identity, isolation, quality, or support floor.


In [ ]:
# -- Your turn: change one commercial driver at a time --------------------
CACHE_REALIZATION = 0.65  # CHANGE THIS: try 0.35 and 0.85.
OUTPUT_TOKEN_CAP = 220  # CHANGE THIS: compare 120, 220, and 600.
SUPPORT_HOURS = 80  # CHANGE THIS: compare 48, 80, and 140 hours/month.

cache_values = expected.copy()
cache_values["cache_realization"] = CACHE_REALIZATION
output_values = expected.copy()
output_values["output_tokens"] = OUTPUT_TOKEN_CAP
support_values = expected.copy()
support_values["support_hours"] = SUPPORT_HOURS

assert 0 <= CACHE_REALIZATION <= 1
assert 1 <= OUTPUT_TOKEN_CAP <= 600
assert SUPPORT_HOURS >= 0
print(f"[Modeled] cache realization {CACHE_REALIZATION:.0%}: ${expected_total(cache_values, expected_rates):,.0f}/month")
print(f"[Modeled] output cap {OUTPUT_TOKEN_CAP}: ${expected_total(output_values, expected_rates):,.0f}/month")
print(f"[Modeled] support hours {SUPPORT_HOURS}: ${expected_total(support_values, expected_rates):,.0f}/month")
print("GUARDRAIL: cache keys retain authorization scope; token caps rerun quality gates; support changes alter the service tier.")

## 8 - Give Riverside a decision, not a deceptive decimal

The current evidence supports discussing a bounded pilot planning range. It does not support a critical service tier or a final quote. The high case crossing the 18,000 USD ceiling triggers exception review, scope reduction, or more evidence; it does not justify quietly changing an assumption.

```mermaid
flowchart TD
    A["Modeled low / expected / high range"] --> D["Decision record"]
    B["Synthetic trace mechanics"] --> D
    C["Policy constraints and unknowns"] --> D
    D --> E{"Every quote prerequisite<br/>has evidence and approval?"}
    E -->|No| F["Pilot, narrow scope,<br/>or collect evidence"]
    E -->|Yes| G["Authorized commercial review"]
```

**Predict before running:** Will the notebook recommend a pilot, a critical tier, or a final quote? Which open evidence items force that result?

| Weak wording | Decision-ready wording |
|---|---|
| "Subject to validation" | "Quota owner supplies regional RPM/TPM evidence by date X" |
| "Expected cost is the price" | "Modeled range expires on date X; finance validates rates and discounts" |
| "99.5% supported" | "Target is limited to named covered hours until load and staffing evidence pass" |

**Plain takeaway:** every condition needs an owner, evidence, deadline, and revalidation trigger.


In [ ]:
# -- Preview and health-check the commercial decision --------------------
decision = {
    "status": "Approve pilot planning envelope; quote pending",
    "recommended_tier": "Pilot",
    "modeled_monthly_range_usd": [round(costs.loc["low", "total"]), round(costs.loc["high", "total"])],
    "expected_case_usd": round(costs.loc["expected", "total"]),
    "operating_target_usd": 14_000,
    "hard_planning_ceiling_usd": 18_000,
    "conditions": ["reconcile session and request demand", "validate request-type distributions under load", "validate price and billing units", "confirm quota and regional capacity", "test rollback/failover", "accept covered hours and severity response"],
}
validation_register = pd.DataFrame([
    ["Demand definitions and burst distribution", "Operations", "Shadow telemetry by request type and tenant tier"],
    ["Token and service-time distributions", "Platform Engineering", "Load test with long-context and retry slices"],
    ["Price, discount, tax, currency, billing unit", "Commercial Lead", "Dated approved price book or quote"],
    ["RPM, TPM, concurrency, regional capacity", "Platform Engineering", "Approved quota evidence per route and region"],
    ["Failover, restore, RTO/RPO", "SRE", "Failure injection and recovery evidence"],
    ["Support hours, severity response, escalation", "Support Lead", "Accepted staffing and service schedule"],
    ["SLA denominator, exclusions, credits, enforceability", "Contract Owner", "Reviewed contract language"],
], columns=["validation", "owner", "required_evidence"])

print(json.dumps(decision, indent=2))
display(validation_register)
print("DECISION: continue only as a bounded pilot estimate; do not issue a critical-tier commitment or quote.")

final_checks = {
    "three_scenarios_present": list(scenarios.index) == ["low", "expected", "high"],
    "average_only_conflict_visible": (scenarios["implied_average_rph"] > scenarios["stated_peak_rph"]).any(),
    "high_case_triggers_ceiling_action": costs.loc["high", "total"] > 18_000,
    "cost_includes_support": costs.loc["expected", "support_engineering"] > 0,
    "capacity_has_independent_limits": {"rpm_utilization", "tpm_utilization", "concurrency_utilization"}.issubset(capacity.columns),
    "critical_tier_withheld": decision["recommended_tier"] != "Critical",
    "external_validations_named": len(validation_register) >= 7,
}
for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
assert all(final_checks.values()), "Stop: the planning package hides a material gap."
print("RESULT: the package is internally reviewable, still modeled, and explicitly not a quote.")

## 9 - Riverside capacity and commercial handoff

```mermaid
flowchart LR
    A["Average-only plan rejected"] --> B["Three demand stories"]
    B --> C["Independent limits checked"]
    C --> D["Whole service priced"]
    D --> E["Sensitive inputs ranked"]
    E --> F["Bounded pilot with conditions"]
```

### Before and after

| Before this notebook | After this notebook |
|---|---|
| 62 sessions/hour was treated like 62 requests/hour | Units conflict is visible; the larger safe planning input is temporary and modeled |
| One expected case looked final | Low, expected, and high stories expose demand, token, retry, and cache assumptions |
| Capacity meant one quota number | RPM, TPM, concurrency, queue, and spend are checked independently |
| Token cost stood in for service cost | Infrastructure, retrieval, observability, software, support, failed attempts, and headroom are included |
| 99.5% looked like a promise | Covered hours, measurement, staffing, rollout, and approval remain explicit |
| Expected cost looked like a quote | Riverside gets a pilot range with quote prerequisites and an expiry |

### Evidence status

| Status | What belongs here |
|---|---|
| `[Measured: synthetic fixture]` | Trace attribution mechanics and fixture integrity checks |
| `[Modeled]` | Demand stories, token/retry/cache effects, concurrency floor, headroom, quota checks, full cost, sensitivity, and tier arithmetic |
| `[External validation required]` | Live prices, discounts, taxes, quota, regional capacity, load tails, cache safety, failover, RTO/RPO, staffing, service credits, and legal language |

### Plain takeaways

1. Keep sessions, requests, tokens, and time units separate.
2. A steady-state concurrency floor does not prove burst or tail behavior.
3. Check requests, tokens, concurrency, queue, and spend independently.
4. Price the operated service, not only successful model tokens.
5. A low/expected/high range is not a probability distribution.
6. Security and duplicate-write constraints remain fail closed; availability cannot trade them away.
7. A target becomes a service commitment only with measurement, architecture, quota, rollout, staffing, and approval.
8. A modeled estimate becomes a quote only after named technical and commercial validation.

**Forward to FDE 06:** carry the pilot tier, cohort caps, rollback conditions, funded support hours, and unresolved evidence owners into rollout planning.
